# Gradient Descent vs. OLS

In Week 04 haben wir das Modell `Sales ~ TV` mit `ols()` geschätzt. `ols()` berechnet die Koeffizienten direkt (OLS). In diesem Notebook schätzen wir **dasselbe Modell** mit **Gradient Descent** und prüfen, ob beide Wege zur **selben Geraden** führen.

*Gradient Descent ist hier nur ein kurzer Ausblick: Für die lineare Regression bleibt OLS der Standard.*

# Arbeitsumgebung einrichten

Pakete importieren.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from statsmodels.formula.api import ols

# Daten laden

Dieselben Daten und dieselbe Aufteilung in Trainings- und Testdaten wie in Week 04.

In [ ]:
path = "https://raw.githubusercontent.com/kbrennig/MODS_WS25_26/refs/heads/main/data/Advertising.csv"
data = pd.read_csv(path)
data_training, data_test = train_test_split(data, test_size=0.2, random_state=42)

Für Gradient Descent speichern wir den Prädiktor TV als `x` und die Zielvariable Sales als `y`.

Mit `.to_numpy()` wandeln wir die beiden Spalten in einfache NumPy-Arrays um. Das ist nicht zwingend nötig, aber die Schleife unten läuft eine Million Mal – mit Arrays geht das Rechnen deutlich schneller als mit pandas-Spalten.

In [ ]:
x = data_training["TV"].to_numpy()
y = data_training["Sales"].to_numpy()

# Modellierung

## OLS: das, was `ols()` in Week 04 gemacht hat

OLS setzt die Ableitungen der RSS gleich 0 und löst die Gleichungen: **ein Schritt, exaktes Ergebnis**.

In [ ]:
model = ols(formula="Sales ~ TV", data=data_training).fit()
beta0_ols = model.params["Intercept"]
beta1_ols = model.params["TV"]
print("OLS: Intercept =", round(beta0_ols, 4), "| Steigung =", round(beta1_ols, 4))

## Gradient Descent: Schritt für Schritt bergab

1. Wir starten mit $\beta_0 = 0$ und $\beta_1 = 0$.
2. Wir berechnen die Steigung der RSS am aktuellen Punkt (die Ableitungen nach $\beta_0$ und $\beta_1$).
3. Wir gehen einen kleinen Schritt bergab: **neuer Wert = alter Wert − Lernrate × Steigung**.
4. Das wiederholen wir sehr oft.

**Lernrate:** Sie bestimmt, wie groß jeder Schritt ist. Ist sie zu klein, brauchen wir sehr viele Schritte. Ist sie zu groß, springen wir über das Minimum hinweg und kommen nie an. Hier muss sie winzig sein (0.0000001), weil die TV-Budgets bis 300 gehen.

*Diese Zelle läuft etwa 10–20 Sekunden.*

In [ ]:
beta0, beta1 = 0.0, 0.0
learning_rate = 0.0000001
history = []

for step in range(1, 1_000_001):
    residual = (beta0 + beta1 * x) - y
    d_beta0 = np.sum(2 * residual)        # Steigung der RSS in Richtung beta0
    d_beta1 = np.sum(2 * x * residual)    # Steigung der RSS in Richtung beta1
    beta0 = beta0 - learning_rate * d_beta0
    beta1 = beta1 - learning_rate * d_beta1
    if step in (10_000, 100_000, 500_000, 1_000_000):
        history.append({"Schritte": step, "Intercept": round(beta0, 4), "Steigung": round(beta1, 4)})

pd.DataFrame(history)

## Vergleich: Beide Wege führen zur selben Geraden

In [ ]:
print("Gradient Descent: Intercept =", round(beta0, 4), "| Steigung =", round(beta1, 4))
print("OLS:              Intercept =", round(beta0_ols, 4), "| Steigung =", round(beta1_ols, 4))

tv = np.linspace(0, 300, 100)
plt.scatter(x, y, color="lightgrey", s=15)
for row in history[:-1]:
    plt.plot(tv, row["Intercept"] + row["Steigung"] * tv, label=f"Gradient Descent, {row['Schritte']:,} Schritte")
plt.plot(tv, beta0_ols + beta1_ols * tv, "k--", linewidth=2, label="OLS")
plt.xlabel("TV")
plt.ylabel("Sales")
plt.legend()
plt.show()

# Zusammenfassung

* **OLS** berechnet das Minimum der RSS direkt: ein Schritt, exakt. Das ist der Standard für die lineare Regression.
* **Gradient Descent** kommt Schritt für Schritt zum selben Minimum. Wie schnell, hängt von der **Lernrate** ab.
* Beide Wege führen zur selben Geraden wie in Week 04. Wichtig ist Gradient Descent für Modelle, bei denen es keine Formel für die besten Koeffizienten gibt.